# Preprocessing T1 and DTI images

In [1]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import pandas as pd
import os
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import networkx as nx
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from tqdm import tqdm
import torch
from torch_geometric.data import Data

In [3]:
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/team3_xai_gnn'
smri_path = os.path.join(base_path, 'oasis3_baseline_freesurfer_dataset.csv')
dmri_path = os.path.join(base_path, 'oasis3_graphmls_scale1')

df = pd.read_csv(smri_path)
graphml_files = [f for f in os.listdir(dmri_path) if f.endswith('.graphml')]

Mounted at /content/drive


# Step 0
## Excluding graphml files with no corresponding session in the csv

In [4]:
def parse_filename(fname):
    match = re.search(r'sub-(OAS3\d+)_ses-(d\d+)', fname)
    if match:
        return match.group(1), match.group(2)
    return None, None

csv_pairs = set(zip(df['subject_id'], df['session_id']))

accepted, rejected = [], []
for f in graphml_files:
    subj, ses = parse_filename(f)
    if (subj, ses) in csv_pairs:
        accepted.append(f)
    else:
        rejected.append(f)

print(f'Accepted: {len(accepted)}, Rejected: {len(rejected)}')

Accepted: 676, Rejected: 299


# Step 1
## Seperating volume and thickness columns

In [5]:
# excluding cols
cols_to_drop = [
    '5th-Ventricle_volume', 'CSF_volume', 'CortexVol', 'CorticalWhiteMatterVol',
    'IntraCranialVol', 'Optic-Chiasm_volume', 'SubCortGrayVol', 'SupraTentorialVol',
    'TOTAL_HIPPOCAMPUS_VOLUME', 'TotalGrayVol', 'WM-hypointensities_volume',
    'lhCortexVol', 'lhCorticalWhiteMatterVol', 'non-WM-hypointensities_volume',
    'rhCortexVol', 'rhCorticalWhiteMatterVol', 'Right-non-WM-hypointensities_volume',
    'Left-non-WM-hypointensities_volume', 'Right-WM-hypointensities_volume',
    'Left-WM-hypointensities_volume', 'L.Numvert', 'R.NumVert', 'L.SurfArea', 'R.SurfArea'
]

# icv is excluded but needed for LR
icv = df['IntraCranialVol'].copy()
df_filtered = df.drop(columns=cols_to_drop)

meta_cols = ['subject_id', 'session_id', 'label', 'age', 'gender']
thickness_cols = [c for c in df_filtered.columns if 'thickness' in c.lower()]
volume_cols = [c for c in df_filtered.columns
               if c not in meta_cols
               and 'thickness' not in c.lower()]

print(f'Volume cols: {len(volume_cols)}')
print(f'Thickness cols: {len(thickness_cols)}')

Volume cols: 104
Thickness cols: 68


### Train - Test Split

In [6]:
train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['label']
)

print(f'Train: {len(train_df)}, Test: {len(test_df)}\n')
print("Train Set")
print(train_df['label'].value_counts().sort_index())
print("\nTest Set")
print(test_df['label'].value_counts().sort_index())

Train: 540, Test: 136

Train Set
label
0    406
1    101
2     33
Name: count, dtype: int64

Test Set
label
0    103
1     25
2      8
Name: count, dtype: int64


### Dividing the dataset into two age groups

In [7]:
print(f'Min age: {df_filtered["age"].min():.1f}')
print(f'Max age: {df_filtered["age"].max():.1f}')

Min age: 42.7
Max age: 95.6


In [8]:
for split_name, split_df in [('Train', train_df), ('Test', test_df)]:
    young = (split_df['age'] < 70).sum()
    old = (split_df['age'] >= 70).sum()
    print(f'{split_name}: 42-69: {young}, 70-96: {old}')

Train: 42-69: 267, 70-96: 273
Test: 42-69: 62, 70-96: 74


# Step 2
## Correcting covariates

In [9]:
def get_age_mask(df, low, high):
    return (df['age'] >= low) & (df['age'] < high)

age_groups = [(42, 70), (70, 96)]

# copy dataframes so we still have the original ones
train_corrected = train_df.copy()
test_corrected = test_df.copy()

### (avoiding type warnings)
train_corrected[volume_cols + thickness_cols] = train_corrected[volume_cols + thickness_cols].astype(float)
test_corrected[volume_cols + thickness_cols] = test_corrected[volume_cols + thickness_cols].astype(float)
###

for feature_cols, predictors in [
    (volume_cols, ['age', 'gender', 'icv']),
    (thickness_cols, ['age', 'gender'])
]:
    # attach ICV temporarily for regression
    train_tmp = train_df.copy()
    test_tmp = test_df.copy()
    train_tmp['icv'] = icv
    test_tmp['icv'] = icv

    for low, high in age_groups:
        train_mask = get_age_mask(train_tmp, low, high)
        test_mask = get_age_mask(test_tmp, low, high)

        # CN subjects in this age group from train set
        train_cn_mask = train_mask & (train_tmp['label'] == 0)
        X_cn = train_tmp.loc[train_cn_mask, predictors]

        for col in feature_cols:
            y_cn = train_tmp.loc[train_cn_mask, col]

            model = LinearRegression()
            model.fit(X_cn, y_cn)

            # predict for everyone in this age group
            train_pred = model.predict(train_tmp.loc[train_mask, predictors])
            test_pred = model.predict(test_tmp.loc[test_mask, predictors])

            # residual = init - pred
            train_corrected.loc[train_mask, col] = (
                train_tmp.loc[train_mask, col].values - train_pred)
            test_corrected.loc[test_mask, col] = (
                test_tmp.loc[test_mask, col].values - test_pred)

print('Covariate correction done.')
print(f'Train corrected shape: {train_corrected.shape}')
print(f'Test corrected shape: {test_corrected.shape}')

Covariate correction done.
Train corrected shape: (540, 177)
Test corrected shape: (136, 177)


# Step 3
## Z-score normalization on T1 data

In [10]:
# Z-score normalization using CN train mean and std
train_cn_corrected = train_corrected[train_corrected['label'] == 0]
feature_cols_to_normalize = volume_cols + thickness_cols

# computing mean and std
cn_mean = train_cn_corrected[feature_cols_to_normalize].mean()
cn_std = train_cn_corrected[feature_cols_to_normalize].std()

# prepping norm vars
train_normalized = train_corrected.copy()
test_normalized = test_corrected.copy()

# normalization
train_normalized[feature_cols_to_normalize] = (train_corrected[feature_cols_to_normalize] - cn_mean) / cn_std
test_normalized[feature_cols_to_normalize] = (test_corrected[feature_cols_to_normalize] - cn_mean) / cn_std

# inspecting results
cn_train_normalized = train_normalized[train_normalized['label'] == 0]
print(f'CN train mean: {cn_train_normalized[feature_cols_to_normalize].mean().mean():.4f}')
print(f'CN train std: {cn_train_normalized[feature_cols_to_normalize].std().mean():.4f}')

# inspecting for test set
# should be slightly off since we used train set mean/std to norm
test_cn_normalized = test_normalized[test_normalized['label'] == 0]
print(f'CN test mean: {test_cn_normalized[feature_cols_to_normalize].mean().mean():.4f}')
print(f'CN test std: {test_cn_normalized[feature_cols_to_normalize].std().mean():.4f}')

CN train mean: 0.0000
CN train std: 1.0000
CN test mean: 0.0219
CN test std: 1.0572


# Step 4
## Z-score normalization on DTI data

In [11]:
# build train/test filename sets from the splits
train_pairs = set(zip(train_df['subject_id'], train_df['session_id']))
test_pairs = set(zip(test_df['subject_id'], test_df['session_id']))

train_files = [f for f in accepted if parse_filename(f) in train_pairs]
test_files = [f for f in accepted if parse_filename(f) in test_pairs]

In [12]:
edge_features_to_drop = ['fiber_density']

def load_graph(filepath):
    G = nx.read_graphml(filepath)
    G.remove_edges_from(nx.selfloop_edges(G))
    for u, v, data in G.edges(data=True):
        for feat in edge_features_to_drop:
            data.pop(feat, None)
    return G

def load_single(f):
    subj, ses = parse_filename(f)
    path = os.path.join(dmri_path, f)
    return (subj, ses), load_graph(path)

# load train graphs in parallel
train_graphs = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = list(tqdm(executor.map(load_single, train_files),
                        total=len(train_files), desc='Loading train graphs'))
train_graphs = dict(futures)

# load test graphs in parallel
test_graphs = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = list(tqdm(executor.map(load_single, test_files),
                        total=len(test_files), desc='Loading test graphs'))
test_graphs = dict(futures)

print(f'\n\nLoaded {len(train_graphs)} train graphs')
print(f'Loaded {len(test_graphs)} test graphs')

sample_graph = next(iter(train_graphs.values()))
sample_edge = next(iter(sample_graph.edges(data=True)))
print(f'\nEdge features after dropping: {list(sample_edge[2].keys())}')
print(f'Number of edge features: {len(sample_edge[2])}')

Loading test graphs: 100%|██████████| 136/136 [01:11<00:00,  1.91it/s]



Loaded 540 train graphs
Loaded 136 test graphs

Edge features after dropping: ['number_of_fibers', 'fiber_length_mean', 'fiber_length_median', 'fiber_length_std', 'fiber_proportion', 'normalized_fiber_density', 'shore_gfa_mean', 'shore_gfa_std', 'shore_gfa_median', 'shore_msd_mean', 'shore_msd_std', 'shore_msd_median', 'shore_rtop_signal_mean', 'shore_rtop_signal_std', 'shore_rtop_signal_median']
Number of edge features: 15


In [13]:
edge_feature_names = [
    'number_of_fibers', 'fiber_length_mean', 'fiber_length_median', 'fiber_length_std',
    'fiber_proportion', 'normalized_fiber_density',
    'shore_gfa_mean', 'shore_gfa_std', 'shore_gfa_median',
    'shore_msd_mean', 'shore_msd_std', 'shore_msd_median',
    'shore_rtop_signal_mean', 'shore_rtop_signal_std', 'shore_rtop_signal_median'
]

# collect edge features from CN train subjects only
cn_train_pairs = set(
    zip(train_df[train_df['label'] == 0]['subject_id'],
        train_df[train_df['label'] == 0]['session_id'])
)

all_cn_train_edges = []
for (subj, ses), G in train_graphs.items():
    if (subj, ses) in cn_train_pairs:
        for u, v, data in G.edges(data=True):
            all_cn_train_edges.append([data[f] for f in edge_feature_names])

all_cn_train_edges = np.array(all_cn_train_edges)

# compute mean and std from CN train set only
edge_mean = all_cn_train_edges.mean(axis=0)
edge_std = all_cn_train_edges.std(axis=0)
print(f'Edge mean shape: {edge_mean.shape}')

# apply z-score to train graphs
for G in train_graphs.values():
    for u, v, data in G.edges(data=True):
        for i, f in enumerate(edge_feature_names):
            data[f] = (data[f] - edge_mean[i]) / edge_std[i]

# apply z-score to test graphs using CN train mean/std
for G in test_graphs.values():
    for u, v, data in G.edges(data=True):
        for i, f in enumerate(edge_feature_names):
            data[f] = (data[f] - edge_mean[i]) / edge_std[i]

print("\nEdge normalization completed")

Edge mean shape: (15,)

Edge normalization completed


In [14]:
'''
add this to inspect node structure

for node_id, attrs in sample_graph.nodes(data=True):
    print(f"{attrs['dn_name'].strip()} | {attrs['dn_region']}")
'''

def get_node_features(node_name, region, row):
    """
    Given a node name, its region type, and a CSV row,
    return a list of features [volume, thickness] or [volume, 0] or [0, 0]
    """
    node_name = node_name.strip()

    # cortical nodes: ctx-rh-* or ctx-lh-*
    if region == 'cortical':
        # ctx-rh-lateralorbitofrontal -> rh_lateralorbitofrontal
        parts = node_name.split('-', 2)  # ['ctx', 'rh', 'lateralorbitofrontal']
        hemisphere = parts[1]            # 'rh' or 'lh'
        region_name = parts[2]           # 'lateralorbitofrontal'

        vol_col = f'{hemisphere}_{region_name}_volume'
        tck_col = f'{hemisphere}_{region_name}_thickness'

        if vol_col in row.index and tck_col in row.index:
            return [row[vol_col], row[tck_col]]
        else:
            return [0.0, 0.0]

    # standard subcortical nodes
    elif region == 'subcortical':
        # handle underscore vs hyphen: Right-Accumbens_area -> Right-Accumbens-area
        vol_col = node_name.replace('_area', '-area') + '_volume'

        if vol_col in row.index:
            return [row[vol_col], 0.0]
        else:
            return [0.0, 0.0]

    return [0.0, 0.0]

# test on the sample graph with the first row of train_normalized
sample_row = train_normalized.iloc[0]
print('Testing node feature mapping on sample graph:')
mapped, unmapped = 0, 0
for node_id, attrs in sample_graph.nodes(data=True):
    name = attrs['dn_name'].strip()
    region = attrs['dn_region']
    feats = get_node_features(name, region, sample_row)
    if feats != [0.0, 0.0]:
        mapped += 1
    else:
        unmapped += 1

print(f'Nodes with features: {mapped}')
print(f'Nodes without features (zeros): {unmapped}')
print(f'Total nodes: {mapped + unmapped}')

Testing node feature mapping on sample graph:
Nodes with features: 82
Nodes without features (zeros): 42
Total nodes: 124


# Step 5
## Attach features to graph nodes

In [15]:
# attach node features to all graphs
def attach_node_features(graphs, normalized_df):
    for (subj, ses), G in tqdm(graphs.items(), desc='Attaching node features'):
        row = normalized_df[(normalized_df['subject_id'] == subj) &
                            (normalized_df['session_id'] == ses)].iloc[0]
        for node_id, attrs in G.nodes(data=True):
            name = attrs['dn_name'].strip()
            region = attrs['dn_region']
            feats = get_node_features(name, region, row)
            attrs['features'] = feats

attach_node_features(train_graphs, train_normalized)
attach_node_features(test_graphs, test_normalized)

# sanity check
sample_graph = next(iter(train_graphs.values()))
print('\n\nSample node features:')
for node_id, attrs in list(sample_graph.nodes(data=True))[:3]:
    print(f"  {attrs['dn_name'].strip()}: {attrs['features']}")

Attaching node features: 100%|██████████| 136/136 [00:00<00:00, 148.99it/s]



Sample node features:
  ctx-rh-lateralorbitofrontal: [np.float64(-0.3964267803544415), np.float64(-0.8240573794822691)]
  ctx-rh-parsorbitalis: [np.float64(-0.10180506736252956), np.float64(-0.0770520768813256)]
  ctx-rh-insula: [np.float64(-0.6199907447629969), np.float64(-0.21658611603132968)]


# Step 6
## Check on results

In [16]:
# graph structure
sample_train_graph = next(iter(train_graphs.values()))
sample_test_graph = next(iter(test_graphs.values()))

print('=== Graph Structure ===')
print(f'Train graphs: {len(train_graphs)}')
print(f'Test graphs:  {len(test_graphs)}')
print(f'Nodes per graph: {sample_train_graph.number_of_nodes()}')
print(f'Edges per graph (sample): {sample_train_graph.number_of_edges()}')

# check no self-edges
self_loops_train = sum(1 for G in train_graphs.values() for u, v in G.edges() if u == v)
self_loops_test = sum(1 for G in test_graphs.values() for u, v in G.edges() if u == v)
print(f'\nSelf-edges in train: {self_loops_train} (should be 0)')
print(f'Self-edges in test:  {self_loops_test} (should be 0)')

=== Graph Structure ===
Train graphs: 540
Test graphs:  136
Nodes per graph: 124
Edges per graph (sample): 3053

Self-edges in train: 0 (should be 0)
Self-edges in test:  0 (should be 0)


In [17]:
# edge features
print('=== Edge Features ===')
sample_edge = next(iter(sample_train_graph.edges(data=True)))
print(f'Edge feature names: {list(sample_edge[2].keys())}')
print(f'Number of edge features: {len(sample_edge[2])} (should be 15)')
print(f'fiber_density present: {"fiber_density" in sample_edge[2]} (should be False)')

# check edge features are z-scored using CN train edges only
cn_check_edges = np.array([[data[f] for f in edge_feature_names]
                            for (subj, ses), G in train_graphs.items()
                            if (subj, ses) in cn_train_pairs
                            for u, v, data in G.edges(data=True)])
print(f'\nCN train edge mean (should be ~0): {cn_check_edges.mean():.4f}')
print(f'CN train edge std  (should be ~1): {cn_check_edges.std():.4f}')

=== Edge Features ===
Edge feature names: ['number_of_fibers', 'fiber_length_mean', 'fiber_length_median', 'fiber_length_std', 'fiber_proportion', 'normalized_fiber_density', 'shore_gfa_mean', 'shore_gfa_std', 'shore_gfa_median', 'shore_msd_mean', 'shore_msd_std', 'shore_msd_median', 'shore_rtop_signal_mean', 'shore_rtop_signal_std', 'shore_rtop_signal_median']
Number of edge features: 15 (should be 15)
fiber_density present: False (should be False)

CN train edge mean (should be ~0): 0.0000
CN train edge std  (should be ~1): 1.0000


In [18]:
# node features
print('=== Node Features ===')
mapped = sum(1 for _, attrs in sample_train_graph.nodes(data=True)
             if attrs['features'] != [0.0, 0.0])
unmapped = sum(1 for _, attrs in sample_train_graph.nodes(data=True)
               if attrs['features'] == [0.0, 0.0])
print(f'Nodes with features: {mapped} (should be 82)')
print(f'Nodes without features: {unmapped} (should be 42)')

# check a cortical and subcortical node
print('\nSample cortical node features:')
for node_id, attrs in sample_train_graph.nodes(data=True):
    if attrs['dn_region'] == 'cortical':
        print(f"  {attrs['dn_name'].strip()}: {attrs['features']}")
        break

print('Sample subcortical node features:')
for node_id, attrs in sample_train_graph.nodes(data=True):
    if attrs['dn_region'] == 'subcortical' and attrs['features'] != [0.0, 0.0]:
        print(f"  {attrs['dn_name'].strip()}: {attrs['features']}")
        break

=== Node Features ===
Nodes with features: 82 (should be 82)
Nodes without features: 42 (should be 42)

Sample cortical node features:
  ctx-rh-lateralorbitofrontal: [np.float64(-0.3964267803544415), np.float64(-0.8240573794822691)]
Sample subcortical node features:
  Right-Putamen: [np.float64(-0.3295222817212704), 0.0]


In [19]:
# label distribution
print('=== Label Distribution ===')
train_labels = [train_normalized[(train_normalized['subject_id'] == subj) &
                (train_normalized['session_id'] == ses)]['label'].values[0]
                for subj, ses in train_graphs.keys()]
test_labels  = [test_normalized[(test_normalized['subject_id'] == subj) &
                (test_normalized['session_id'] == ses)]['label'].values[0]
                for subj, ses in test_graphs.keys()]

import collections
print(f'Train: {dict(sorted(collections.Counter(train_labels).items()))}')
print(f'Test:  {dict(sorted(collections.Counter(test_labels).items()))}')

=== Label Distribution ===
Train: {np.int64(0): 406, np.int64(1): 101, np.int64(2): 33}
Test:  {np.int64(0): 103, np.int64(1): 25, np.int64(2): 8}


# Step 7
## Save graphs for the next notebook

In [20]:
def nx_to_pyg(G, label):
    # node features
    node_list = list(G.nodes(data=True))
    x = torch.tensor([attrs['features'] for _, attrs in node_list], dtype=torch.float)

    # node index mapping
    node_idx = {node_id: i for i, (node_id, _) in enumerate(node_list)}

    # edge index and edge features
    edge_index = []
    edge_attr = []
    # add directed edges to both directions for msg passing
    for u, v, data in G.edges(data=True):
      edge_index.append([node_idx[u], node_idx[v]])
      edge_index.append([node_idx[v], node_idx[u]])
      edge_attr.append([data[f] for f in edge_feature_names])
      edge_attr.append([data[f] for f in edge_feature_names])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    y = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

# convert train graphs
train_dataset = []
for (subj, ses), G in tqdm(train_graphs.items(), desc='Converting train graphs'):
    label = train_normalized[(train_normalized['subject_id'] == subj) &
                              (train_normalized['session_id'] == ses)]['label'].values[0]
    train_dataset.append(nx_to_pyg(G, label))

# convert test graphs
test_dataset = []
for (subj, ses), G in tqdm(test_graphs.items(), desc='Converting test graphs'):
    label = test_normalized[(test_normalized['subject_id'] == subj) &
                             (test_normalized['session_id'] == ses)]['label'].values[0]
    test_dataset.append(nx_to_pyg(G, label))

# save
torch.save(train_dataset, '/content/drive/MyDrive/team3_xai_gnn/train_dataset.pt')
torch.save(test_dataset, '/content/drive/MyDrive/team3_xai_gnn/test_dataset.pt')

print(f'\nSaved {len(train_dataset)} train graphs and {len(test_dataset)} test graphs')

Converting test graphs: 100%|██████████| 136/136 [00:07<00:00, 18.50it/s]



Saved 540 train graphs and 136 test graphs
